# Notebook 04 — Spectral Analysis and Feature Engineering

## Purpose
Compute graph-theoretic features rooted in spectral graph theory: PageRank 
(full graph and fraud subgraph, compared against in-degree), and Laplacian 
eigenvalue-based features (Fiedler value, connected components, spectral 
gap) on the fraud subgraph versus a comparable normal subgraph.

## Inputs
- `data/processed/transaction_graph.pkl`

## Key deferred decision (from notebook 02)
Compute PageRank on both full graph and fraud subgraph; compare against 
in-degree to test for independent signal; keep or drop based on evidence.

In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pickle
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh

with open('../data/processed/transaction_graph.pkl', 'rb') as f:
    G = pickle.load(f)

print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

Nodes: 3,277,489
Edges: 2,770,393


In [4]:
#PageRank on the full transction graph

pagerank_full = nx.pagerank(G, alpha=0.85, weight='weight')

pr_values = np.array(list(pagerank_full.values()))
print(f"PageRank computed for {len(pagerank_full):,} nodes")
print(f"Mean: {pr_values.mean():.2e}")
print(f"Max:  {pr_values.max():.2e}")
print(f"Min:  {pr_values.min():.2e}")

PageRank computed for 3,277,489 nodes
Mean: 3.05e-07
Max:  1.95e-05
Min:  8.60e-08


In [5]:
#Compare PageRanks against in_degree
in_degree = dict(G.in_degree())

nodes = list(G.nodes())
pr_array = np.array([pagerank_full[n] for n in nodes])
indeg_array = np.array([in_degree[n] for n in nodes])

# Correlation btween PageRank and in_degree

correlation = np.corrcoef(pr_array, indeg_array)[0,1]
print(f"Pearson correlation (PageRank, in-degree): {correlation:.4f}")

# Spearman correlation — rank based, more robust
from scipy.stats import spearmanr
spearman_corr, p_value = spearmanr(pr_array, indeg_array)
print(f"Spearman correlation (PageRank, in-degree): {spearman_corr:.4f}")
print(f"P-value: {p_value:.2e}")

Pearson correlation (PageRank, in-degree): 1.0000
Spearman correlation (PageRank, in-degree): 1.0000
P-value: 0.00e+00


In [6]:
# Build the fraud subgraph : fraud edges plus their endpoint nodes

fraud_edges = [(u,v) for u, v, d in G.edges(data=True) if d['is_fraud'] == 1]
G_fraud = nx.DiGraph()
G_fraud.add_edges_from(fraud_edges)

print(f"Fraud subgraph nodes: {G_fraud.number_of_nodes():,}")
print(f"Fraud subgraph edges: {G_fraud.number_of_edges():,}")
print(f"Is connected (weakly): {nx.is_weakly_connected(G_fraud)}")
print(f"Number of weakly connected components: {nx.number_weakly_connected_components(G_fraud)}")

Fraud subgraph nodes: 16,351
Fraud subgraph edges: 8,197
Is connected (weakly): False
Number of weakly connected components: 8154


In [7]:
# Check the actual edge attributes
sample_edge = list(G.edges(data=True))[0]
print(sample_edge)

('C1305486145', 'C553264065', {'weight': 181.0, 'is_fraud': 1, 'transaction_type': 'TRANSFER', 'step': 1})


In [8]:
# PageRank on the fraud subgraph
pagerank_fraud = nx.pagerank(G_fraud, alpha=0.85)

# In degree within th fraud subgrapgh
fraud_in_degree = dict(G_fraud.in_degree())

fraud_nodes = list(G_fraud.nodes())
pr_fraud_array = np.array([pagerank_fraud[n] for n in fraud_nodes ])
indeg_fraud_array = np.array([fraud_in_degree[n] for n in fraud_nodes])

corr_fraud = np.corrcoef(pr_fraud_array,indeg_fraud_array)[0,1]
spearman_fraud, p_fraud = spearmanr(pr_fraud_array, indeg_fraud_array)

print(f"Pearson correlation (fraud subgraph): {corr_fraud:.4f}")
print(f"Spearman correlation (fraud subgraph): {spearman_fraud:.4f}")

Pearson correlation (fraud subgraph): 1.0000
Spearman correlation (fraud subgraph): 1.0000


PageRank fully tested: full graph AND fraud subgraph, both r=1.0000 with in-degree
Reason: graph structure prevents multi-hop propagation — 88% of nodes have 
degree 1, fraud subgraph is 8,154 disconnected star-shaped clusters
Decision: PageRank dropped from feature set, in-degree retained

In [ ]:
# Symmetrize the fraud subgraph (undirected version for spectral analysis)
G_fraud_undirected = G_fraud.to_undirected()

print(f"Undiredted fraud subgraph nodes: {G_fraud_undirected.number_of_nodes():,}")
print(f"Undiredted fraud subgraph edges: {G_fraud_undirected.number_of_edges():,}")
print(f"Number of connected components: {nx.number_connected_components(G_fraud_undirected)}")

Undiredtd fraud subgraph nodes: 16,351
Undiredtd fraud subgraph edges: 8,197
Number of connected components: 8154


In [10]:
# Compute the Laplacuian matrix and its eigenvalues

L_fraud = nx.laplacian_matrix(G_fraud_undirected).astype(float)
print(f"Laplacian matrix shape: {L_fraud.shape}")
print(f"Non- zero entries:{L_fraud.nnz}")

# Compute the smallest eigenvalues using sparsing solver
# We ask for more than 8154 to see where the spectrum departs from zero

num_eigenvalues = 8160
eigenvalues = eigsh(L_fraud, k=num_eigenvalues, which='SM', return_eigenvectors=False)
eigenvalues_sorted =np.sort(eigenvalues)
zero_count = np.sum(eigenvalues_sorted < 1e-8)

print(f"\nEigenvalues computed: {len(eigenvalues_sorted)}")
print(f"Number effectively zero (< 1e-8): {zero_count}")
print(f"Expected (connected components): 8154")
print(f"\nSmallest 5 eigenvalues: {eigenvalues_sorted[:5]}")
print(f"Eigenvalues 8150-8160: {eigenvalues_sorted[8150:8160]}")



Laplacian matrix shape: (16351, 16351)


Non- zero entries:32745

Eigenvalues computed: 8160
Number effectively zero (< 1e-8): 8124
Expected (connected components): 8154

Smallest 5 eigenvalues: [-8.03052802e-15 -2.04015357e-15 -1.88735987e-15 -1.77806277e-15
 -1.72967837e-15]
Eigenvalues 8150-8160: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


Laplacian eigenvalue verification: 8124/8154 zero eigenvalues found (99.6% match)
Discrepancy explained: eigsh (Lanczos iterative solver) struggles with highly 
degenerate spectra — many exactly-equal eigenvalues are harder to individually 
resolve than distinct ones
Eigenvalues 8150-8160 all = 1.0, consistent with star-graph structure of 
fraud subgraph clusters
Confirms structural hypothesis: fraud subgraph = many disconnected star clusters

In [11]:
# Build a comparably_sized normal (non-fraud) sub-graph for comparison
normal_edges =[(u,v) for u, v, d in G.edges(data=True) if d.get('is_fraud', 0) == 0]

# Randomly sample the same the number of edges as the fraud subgraph
np.random.seed(42)
sample_idx = np.random.choice(len(normal_edges), size = len(fraud_edges), replace = False)
normal_sample_edges = [normal_edges[i] for i in sample_idx]
 
G_normal_sample = nx.DiGraph()
G_normal_sample.add_edges_from(normal_sample_edges)
G_normal_undirected = G_normal_sample.to_undirected()

print(f"Normal sample nodes: {G_normal_undirected.number_of_nodes():,}")
print(f"Normal sample edges: {G_normal_undirected.number_of_edges():,}")
print(f"Connected components: {nx.number_connected_components(G_normal_undirected)}")

# Compare component size distributions
fraud_components = list(nx.connected_components(G_fraud_undirected))
normal_components = list(nx.connected_components(G_normal_undirected))

fraud_sizes = [len(c) for c in fraud_components]
normal_sizes = [len(c) for c in normal_components]

print(f"\nFraud subgraph — avg component size: {np.mean(fraud_sizes):.2f}, max: {max(fraud_sizes)}")
print(f"Normal sample — avg component size: {np.mean(normal_sizes):.2f}, max: {max(normal_sizes)}")


Normal sample nodes: 16,277
Normal sample edges: 8,197
Connected components: 8080

Fraud subgraph — avg component size: 2.01, max: 3
Normal sample — avg component size: 2.01, max: 3


Component size comparison: fraud avg=2.01/max=3, normal avg=2.01/max=3
Identical — component-level structure does NOT discriminate fraud from normal
Consistent with notebook 01 finding: 88% of ALL nodes (not just fraud) have degree 1
Conclusion: degree asymmetry (in-degree specifically) remains the strongest 
structural signal; spectral component features add nothing at local scale

In [12]:
# Spectral gap analysis — since components are tiny (max size 3), 
# compute this per-component rather than on the whole disconnected graph
# (a globally disconnected graph trivially has zero spectral gap everywhere)

def component_spectral_gap(G_sub, component):
    """For a small component, return its smallest non-zero Laplacian eigenvalue"""
    subG = G_sub.subgraph(component)
    if len(component) < 2:
        return None
    L_sub = nx.laplacian_matrix(subG).astype(float).toarray()
    eigvals = np.linalg.eigvalsh(L_sub)
    eigvals_sorted = np.sort(eigvals)
    # First eigenvalue is always ~0, second is the spectral gap
    nonzero = eigvals_sorted[eigvals_sorted > 1e-8]
    return nonzero[0] if len(nonzero) > 0 else 0

fraud_gaps = [component_spectral_gap(G_fraud_undirected, c) 
              for c in fraud_components if len(c) >= 2]
normal_gaps = [component_spectral_gap(G_normal_undirected, c) 
               for c in normal_components if len(c) >= 2]

print(f"Fraud component spectral gaps — mean: {np.mean(fraud_gaps):.4f}, "
      f"unique values: {sorted(set(np.round(fraud_gaps, 2)))}")
print(f"Normal component spectral gaps — mean: {np.mean(normal_gaps):.4f}, "
      f"unique values: {sorted(set(np.round(normal_gaps, 2)))}")

Fraud component spectral gaps — mean: 1.9947, unique values: [np.float64(1.0), np.float64(2.0)]
Normal component spectral gaps — mean: 1.9855, unique values: [np.float64(1.0), np.float64(2.0)]


# Notebook 04 — Summary of Findings

## What we tested
Following the deferred decision from notebook 02, this notebook rigorously 
tested whether spectral graph theory methods add discriminative signal 
beyond the degree-based features already found in notebook 01.

## Results

**PageRank** (full graph and fraud subgraph): Pearson and Spearman 
correlation with in-degree = 1.0000 in both cases. Mathematically redundant 
given this network's structure — 88% of all nodes have degree 1, and the 
fraud subgraph consists of 8,154 disconnected star-shaped clusters with no 
multi-hop paths for importance to propagate through. **Dropped.**

**Laplacian connected-components theorem**: verified empirically — 8,124 of 
an expected 8,154 zero eigenvalues found via sparse iterative solver (99.6% 
match; discrepancy is a known numerical artefact of degenerate spectra, not 
an error in the theory).

**Component size and spectral gap**: statistically identical between fraud 
and normal subgraphs (avg component size 2.01 in both; spectral gap values 
drawn from the same discrete set {1.0, 2.0} in both). No discriminative 
power — both networks are dominated by simple 2-3 node star components.

## Conclusion
Spectral graph theory, while theoretically elegant and successfully verified 
against known theorems on real data, does not add discriminative signal in 
this specific network. This is a structural consequence of the transaction 
graph's sparsity (most accounts transact once) rather than a limitation of 
the theory itself — spectral methods are most powerful on graphs with richer 
multi-hop connectivity, which this network largely lacks.

## Features carried forward to modelling
- In-degree, out-degree (from notebook 02) — remain the strongest 
  structural signal
- Transaction amount (from notebook 01)
- Temporal cycle position (from notebook 03)

## Features tested and explicitly excluded
- PageRank (full graph and fraud subgraph)
- Component size
- Spectral gap